# Argoverse 1 Motion Forecasting — Exploration

This notebook loads one scenario from the Argoverse 1 Motion Forecasting dataset (via Kaggle), visualizes trajectories, and runs the preprocessing pipeline. Designed to run in **Google Colab** or locally.

## 1. Setup (Colab: add project to path and install deps)

In [ ]:
# Clone project (Colab: run this first). If repo exists, clean cache and pull branch.
import os
import sys

REPO_URL = "https://github.com/PulockDas/Implement-STAST-System.git"  # set your repo URL
BRANCH = "master"  # branch to checkout
CLONE_DIR = "/content/Implement-STAST-System"  # Colab; use os.getcwd() for local

ip = get_ipython()
if os.path.isdir(CLONE_DIR) and os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    ip.run_line_magic("cd", CLONE_DIR)
    ip.system("git fetch origin")
    ip.system("git clean -fd")
    ip.system(f"git checkout {BRANCH}")
    ip.system(f"git pull origin {BRANCH}")
    print(f"Updated existing repo at {CLONE_DIR} (branch {BRANCH})")
else:
    parent = os.path.dirname(CLONE_DIR) or "."
    os.makedirs(parent, exist_ok=True)
    ip.run_line_magic("cd", parent)
    ip.system(f"git clone --branch {BRANCH} {REPO_URL} {os.path.basename(CLONE_DIR)}")
    print(f"Cloned repo to {CLONE_DIR} (branch {BRANCH})")

PROJECT_ROOT = CLONE_DIR
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
ip.run_line_magic("cd", PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# If you didn't run the clone cell (e.g. local): set project root to parent of notebooks/.
import sys
import os

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# Install dependencies (skip if already installed, e.g. in Colab)
!pip install -q tqdm

## 2. Download dataset with kagglehub

In [ ]:
import kagglehub

# Download Argoverse 1 Motion Dataset from Kaggle
path = kagglehub.dataset_download("narendarmallireddy/argoverse1-motion-dataset")
print("Dataset path:", path)

In [ ]:
# Find scenario CSV files (layout may vary: train/val/test or flat)
from pathlib import Path

data_path = Path(path)
csv_files = list(data_path.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s).")
if csv_files:
    # Use first CSV as example
    scenario_path = str(csv_files[0])
    print("Example scenario:", scenario_path)
else:
    raise FileNotFoundError("No CSV files found under dataset path. Check dataset layout.")

## 3. Load one scenario

In [ ]:
from datasets.argoverse_loader import load_scenario_from_path

scenario = load_scenario_from_path(scenario_path, include_city=True)
trajectories = scenario["trajectories"]
print(f"Number of tracks: {len(trajectories)}")
for i, t in enumerate(trajectories[:3]):
    print(f"  Track {i}: track_id={t['track_id']}, object_type={t['object_type']}, "
          f"len(trajectory)={len(t['trajectory'])}, len(timestamps)={len(t['timestamps'])}")

In [ ]:
# Show one trajectory structure (output format)
example = trajectories[0]
print("Example trajectory dict keys:", list(example.keys()))
print("First 3 points (x,y):", example["trajectory"][:3])
print("First 3 timestamps:", example["timestamps"][:3])

## 4. Visualize trajectories (past vs future, highlight target)

In [ ]:
import matplotlib.pyplot as plt
from visualization.plot_trajectories import plot_scenario_trajectories
from utils.config import PAST_STEPS, FUTURE_STEPS

# Highlight AGENT if present, else first track
target_id = None
for t in trajectories:
    if t.get("object_type") == "AGENT":
        target_id = t["track_id"]
        break
if target_id is None and trajectories:
    target_id = trajectories[0]["track_id"]

fig = plot_scenario_trajectories(
    trajectories,
    past_steps=PAST_STEPS,
    future_steps=FUTURE_STEPS,
    target_track_id=target_id,
    title="Argoverse scenario — past (blue) vs future (coral), target in bold"
)
plt.show()

## 5. Preprocessing: past/future extraction and PyTorch tensors

In [ ]:
from preprocessing.trajectory_processor import prepare_agent_tensors
from utils.config import PAST_STEPS, FUTURE_STEPS, FEATURE_DIM_POSITION, FEATURE_DIM_WITH_VELOCITY

out = prepare_agent_tensors(
    trajectories,
    past_steps=PAST_STEPS,
    future_steps=FUTURE_STEPS,
    add_velocity=True,
    device=None,
)

past_t = out["past_tensor"]
future_t = out["future_tensor"]
print("past_tensor shape:", past_t.shape)
print("future_tensor shape:", future_t.shape)
print("Expected: past [num_agents, past_steps, features], future [num_agents, future_steps, 2]")
print("Features: past includes (x,y,vx,vy) ->", FEATURE_DIM_WITH_VELOCITY, "; future (x,y) ->", FEATURE_DIM_POSITION)
print("Track IDs:", out["track_ids"])
print("Object types:", out["object_types"])

In [ ]:
# Quick sanity check: first agent past positions vs raw trajectory
import numpy as np
first_past = out["past_list"][0]
first_traj = [t for t in trajectories if t["track_id"] == out["track_ids"][0]][0]["trajectory"]
np.testing.assert_allclose(first_past[:, :2], np.array(first_traj[:PAST_STEPS]), err_msg="Past positions should match")
print("Sanity check passed: past positions match raw trajectory.")

## 6. Summary

- **Dataset loader**: `load_scenario_from_path(csv_path)` returns trajectories grouped by `TRACK_ID` with `trajectory`, `timestamps`, `object_type`.
- **Preprocessing**: `prepare_agent_tensors(trajectories)` yields `past_tensor` and `future_tensor` ready for PyTorch models.
- **Visualization**: `plot_scenario_trajectories(trajectories, target_track_id=...)` plots past vs future and highlights the target vehicle.